# EnSIMem: LoCoMo loader and grounded memory lookup

This notebook accompanies [Meng et al., arXiv:2609.27279](https://arxiv.org/abs/2609.27279). It loads the public LoCoMo benchmark and demonstrates EnSIMem's entity/property address plus source-evidence expansion without using an LLM or API key.

In [ ]:
import json
import re
import urllib.request
from pprint import pprint

## 1. Load the public LoCoMo dataset

LoCoMo contains ten long conversations with QA items that point back to dialogue-turn evidence IDs. The official data is distributed under CC BY-NC 4.0.

In [ ]:
LOCOMO_URL = 'https://raw.githubusercontent.com/snap-research/locomo/main/data/locomo10.json'
with urllib.request.urlopen(LOCOMO_URL) as response:
    locomo = json.load(response)

print(f'Loaded {len(locomo)} long conversations.')
print('First two QA examples:')
for row in locomo[0]['qa'][:2]:
    pprint(row)

## 2. Recover source turns by provenance ID

The first conversation stores sessions under keys such as `session_1`, and each turn has a stable `dia_id` such as `D1:3`.

In [ ]:
conversation = locomo[0]['conversation']
session_keys = sorted(
    (key for key in conversation if re.fullmatch(r'session_\d+', key)),
    key=lambda key: int(key.split('_')[1]),
)
turn_by_id = {
    turn['dia_id']: {
        **turn,
        'session': session_key,
        'session_date_time': conversation.get(f'{session_key}_date_time'),
    }
    for session_key in session_keys
    for turn in conversation[session_key]
}

for source_id in ('D1:3', 'D2:8'):
    pprint(turn_by_id[source_id])

## 3. Build a tiny dialogue-grounded index

These two records are manually extracted from the loaded source turns. Each compact address retains its source ID; the original dialogue remains the evidence.

In [ ]:
memory_index = [
    {
        'entity': 'Caroline',
        'entity_type': 'person',
        'properties': {'attendance': 'LGBTQ support group'},
        'episode_id': 'session_1',
        'source_ids': ['D1:3'],
    },
    {
        'entity': 'Caroline',
        'entity_type': 'person',
        'properties': {'research': 'adoption agencies'},
        'episode_id': 'session_2',
        'source_ids': ['D2:8'],
    },
]
pprint(memory_index)

## 4. Plan, retrieve, and expand evidence

The point query requires one entity/property pair. Structured lookup selects the address, after which the notebook restores the original supporting turn.

In [ ]:
query = 'What did Caroline research?'
query_plan = {
    'entity': 'Caroline',
    'entity_type': 'person',
    'property': 'research',
    'reasoning_type': 'point',
    'required_records': 1,
}

matches = [
    record
    for record in memory_index
    if record['entity'].casefold() == query_plan['entity'].casefold()
    and query_plan['property'].casefold() in {key.casefold() for key in record['properties']}
][:query_plan['required_records']]

evidence = [turn_by_id[source_id] for record in matches for source_id in record['source_ids']]
answer_values = [record['properties'][query_plan['property']] for record in matches]

print('Query:')
print(query)
print('\nRequirement plan:')
pprint(query_plan)
print('\nMatched index address:')
pprint(matches)
print('\nExpanded source evidence:')
pprint(evidence)
print('\nAnswer from grounded value:')
print(', '.join(answer_values))

### Scope

This notebook demonstrates the paper's core data structure and provenance path. It does not reproduce the reported benchmark scores or the LLM-based construction/planning pipeline. For that, use the authors' official EnSIMem repository and its benchmark-specific reproduction instructions.